# BLOQUE 01 – PERFIL DEMOGRÁFICO - CONDICIONANTES
## Objetivo
Construir capa consolidada demográfica normalizada por unidad espacial.
## Inputs
- manzanas.gpkg
- censo_2020.csv
- conapo.xlsx
## Procesos
- Limpieza
- Homologación de CRS
- Join por CVEGEO

## Output
-D2C1V1._output_perfil_demografico.gpkg

In [ ]:
import pandas as pd
import geopandas as gpd
import os

In [ ]:
# 1. RUTAS DE ENTRADA (Donde están tus datos base)
archivo_censo = r"C:\Users\denih\Downloads\dal\data_raw\censo_2020.csv"
archivo_conapo = r"C:\Users\denih\Downloads\dal\data_raw\conapo.xlsx"
archivo_manzanas = r"C:\Users\denih\Downloads\dal\data_raw\manzanas.gpkg"

In [ ]:
# 2. RUTA DE SALIDA (El archivo donde quieres consolidar todo)
ruta_output_gpkg = r"C:\Users\denih\Downloads\dal\outputs\D2C1V1.gpkg"

In [ ]:
# --- PROCESAMIENTO ---

# Leer Censo
MZNA_2020_df = pd.read_csv(os.path.join(ruta_input, archivo_censo), dtype=str, encoding='latin-1')
cols_discapacidad = ['PCON_DISC', 'PCDISC_MOT', 'PCDISC_VIS', 'PCDISC_LENG', 'PCDISC_AUD', 'PCDISC_MOT2', 'PCDISC_MEN']

MZNA_2020_df['CVEGEO'] = (
    MZNA_2020_df['ENTIDAD'].str.zfill(2) + MZNA_2020_df['MUN'].str.zfill(3) + 
    MZNA_2020_df['LOC'].str.zfill(4) + MZNA_2020_df['AGEB'].str.zfill(4) + 
    MZNA_2020_df['MZA'].str.zfill(3)
)

# Leer CONAPO (Posición 11 -> índice 10)
conapo = pd.read_excel(os.path.join(ruta_input, archivo_conapo), dtype=str)
conapo['CVE_AGEB'] = conapo.iloc[:, 7].str.replace(r'\.0$', '', regex=True).str.zfill(13)
conapo_clean = conapo[['CVE_AGEB', conapo.columns[10], conapo.columns[27]]].copy()
conapo_clean.columns = ['CVE_AGEB', 'POB_ANALFABETA', 'GRADO_REZAGO']

# Leer Manzanas del Input y Filtrar AMG
manzanas = gpd.read_file(os.path.join(ruta_input, archivo_manzanas))
manzanas['CVEGEO'] = manzanas['CVEGEO'].astype(str).str.strip()
municipios_amg = ['039', '120', '098', '101', '097', '070', '044', '051', '124']
manzanas_amg = manzanas[manzanas['CVEGEO'].str[2:5].isin(municipios_amg)].copy()
manzanas_amg['CVE_AGEB'] = manzanas_amg['CVEGEO'].str[:13]

In [25]:
# Joins
paso1 = manzanas_amg.merge(MZNA_2020_df[['CVEGEO'] + cols_discapacidad], on='CVEGEO', how='left')
D2C1V2 = paso1.merge(conapo_clean, on='CVE_AGEB', how='left')

# --- GUARDADO ESTRATÉGICO ---

# 3. GUARDAR EN EL ARCHIVO DE OUTPUTS COMO UNA NUEVA CAPA
# Si el archivo D2C1V1.gpkg ya existe, esto añade la capa D2C1V2 sin borrar la anterior
D2C1V2.to_file(ruta_output_gpkg, layer='D2C1V2', driver="GPKG")

print(f"--- PROCESO TERMINADO ---")
print(f"La capa 'D2C1V2' se guardó correctamente dentro de:\n{ruta_output_gpkg}")

C:\Users\denih\anaconda3\Lib\site-packages\pyogrio\geopandas.py:265: UserWarning: More than one layer found in 'manzanas.gpkg': 'manzanas' (default), 'D2C1V2'. Specify layer parameter to avoid this warning.
  result = read_func(


--- PROCESO TERMINADO ---
La capa 'D2C1V2' se guardó correctamente dentro de:
C:\Users\denih\Downloads\dal\outputs\D2C1V1.gpkg
